# Desafío: Analizando texto sobre Ciencia de Datos

En este ejemplo, hagamos un ejercicio simple que cubre todos los pasos de un proceso tradicional de ciencia de datos. No tienes que escribir ningún código, puedes simplemente hacer clic en las celdas a continuación para ejecutarlas y observar el resultado. Como desafío, se te anima a probar este código con diferentes datos.

## Objetivo

En esta lección, hemos estado discutiendo diferentes conceptos relacionados con la Ciencia de Datos. Intentemos descubrir más conceptos relacionados haciendo un **análisis de texto**. Empezaremos con un texto sobre Ciencia de Datos, extraeremos palabras clave de él y luego intentaremos visualizar el resultado.

Como texto, usaré la página sobre Ciencia de Datos de Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Paso 1: Obtener los Datos

El primer paso en todo proceso de ciencia de datos es obtener los datos. Usaremos la biblioteca `requests` para eso:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Paso 2: Transformando los Datos

El siguiente paso es convertir los datos en la forma adecuada para el procesamiento. En nuestro caso, hemos descargado el código fuente HTML de la página, y necesitamos convertirlo en texto plano.

Hay muchas formas de hacer esto. Usaremos [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), una biblioteca popular de Python para analizar HTML. BeautifulSoup nos permite apuntar a elementos HTML específicos, para que podamos centrarnos en el contenido principal del artículo de Wikipedia y reducir algunos menús de navegación, barras laterales, pies de página y otros contenidos irrelevantes (aunque puede que aún quede algo de texto estándar).


Primero, necesitamos instalar la biblioteca BeautifulSoup para el análisis HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Paso 3: Obtener ideas

El paso más importante es convertir nuestros datos en alguna forma de la cual podamos extraer ideas. En nuestro caso, queremos extraer palabras clave del texto y ver cuáles son más significativas.

Usaremos la biblioteca de Python llamada [RAKE](https://github.com/aneesha/RAKE) para la extracción de palabras clave. Primero, instalemos esta biblioteca en caso de que no esté presente:


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

La funcionalidad principal está disponible desde el objeto `Rake`, que podemos personalizar utilizando algunos parámetros. En nuestro caso, estableceremos la longitud mínima de una palabra clave en 5 caracteres, la frecuencia mínima de una palabra clave en el documento en 3 y el número máximo de palabras en una palabra clave en 2. Siéntete libre de experimentar con otros valores y observar el resultado.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Obtuvimos una lista de términos junto con su grado de importancia asociado. Como puede ver, las disciplinas más relevantes, como el aprendizaje automático y big data, están presentes en la lista en las posiciones superiores.

## Paso 4: Visualización del Resultado

Las personas pueden interpretar mejor los datos en forma visual. Por lo tanto, a menudo tiene sentido visualizar los datos para obtener algunos insights. Podemos usar la biblioteca `matplotlib` en Python para graficar la distribución simple de las palabras clave con su relevancia:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Sin embargo, hay una forma aún mejor de visualizar las frecuencias de palabras: usando **Nube de Palabras**. Necesitaremos instalar otra biblioteca para graficar la nube de palabras a partir de nuestra lista de palabras clave.


In [ ]:
!{sys.executable} -m pip install wordcloud

El objeto `WordCloud` es responsable de tomar ya sea texto original o una lista precomputada de palabras con sus frecuencias, y devuelve una imagen, que luego puede mostrarse usando `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

También podemos pasar el texto original a `WordCloud`: veamos si podemos obtener un resultado similar:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Puedes ver que la nube de palabras ahora se ve más impresionante, pero también contiene mucho ruido (por ejemplo, palabras no relacionadas como `Retrieved on`). Además, obtenemos menos palabras clave que consisten en dos palabras, como *data scientist* o *computer science*. Esto se debe a que el algoritmo RAKE hace un trabajo mucho mejor al seleccionar buenas palabras clave del texto. Este ejemplo ilustra la importancia del preprocesamiento y limpieza de datos, porque una imagen clara al final nos permitirá tomar mejores decisiones.

En este ejercicio hemos recorrido un proceso sencillo de extraer algo de significado del texto de Wikipedia, en forma de palabras clave y nube de palabras. Este ejemplo es bastante simple, pero demuestra bien todos los pasos típicos que un científico de datos seguirá al trabajar con datos, desde la adquisición de datos, hasta la visualización.

En nuestro curso discutiremos todos esos pasos en detalle.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
